<a href="https://colab.research.google.com/github/rylan-berry/DeepLearningIndependentStudy/blob/main/AttentionBlocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python3 -m pip install --upgrade rb-deeplearning-lib
import rb_deeplearning_lib as rb

In [2]:
from rb_deeplearning_lib import Values, Model, mse_loss

#Head
x = [seq_len, dims]

w_q = [dims,dims]

w_k = [dims,dims]

w_v = [dims,dims]



Q = x @ w_q

K = x @ w_k

V = x @ w_v



S = (Q @ K.T)/(sqrt(dims))

A = S.softmax()

Y = A @ V


In [3]:
import numpy as np

In [4]:
class AttentionHead():
  def __init__(self, dims,rangeQ=(-1,1),rangeK=(-1,1),rangeV=(-1,1)):
    self.w_q = Values((rangeQ[0]-rangeQ[1])*np.random.rand(dims,dims)+rangeQ[1])
    self.w_k = Values((rangeK[0]-rangeK[1])*np.random.rand(dims,dims)+rangeK[1])
    self.w_v = Values((rangeV[0]-rangeV[1])*np.random.rand(dims,dims)+rangeV[1])
    self.dims = dims

  def __call__(self, x):
    Q = x @ self.w_q
    K = x @ self.w_k
    V = x @ self.w_v

    S = (Q @ K.transpose((-1, -2)))/(self.dims**0.5)
    A = S.softmax()
    Y = A @ V

    return Y

In [5]:
#test of AttentionHead
dims = 64
seq_len = 10
attention_head = AttentionHead(dims)
x_input = Values(np.random.rand(seq_len, dims))
output = attention_head(x_input)

print(f"Input shape: {x_input.shape}")
print(f"Output shape: {output.shape}")

Input shape: (10, 64)
Output shape: (10, 64)


In [6]:
class AttentionHead():
  def __init__(self, dims1,dims2=None,rangeQ=(-1,1),rangeK=(-1,1),rangeV=(-1,1)):
    if not dims2:
      dims2 = dims1

    self.w_q = Values((rangeQ[0]-rangeQ[1])*np.random.rand(dims1,dims2)+rangeQ[1])
    self.w_k = Values((rangeK[0]-rangeK[1])*np.random.rand(dims1,dims2)+rangeK[1])
    self.w_v = Values((rangeV[0]-rangeV[1])*np.random.rand(dims1,dims2)+rangeV[1])
    self.dims = dims2

  def __call__(self, x):
    Q = x @ self.w_q
    K = x @ self.w_k
    V = x @ self.w_v

    axes = list(range(K.vals.ndim))
    axes[-1], axes[-2] = axes[-2], axes[-1]

    S = (Q @ K.transpose(tuple(axes)))/(self.dims**0.5)
    A = S.softmax()
    Y = A @ V

    return Y

  def params(self):
    return self.w_q, self.w_k, self.w_v

In [7]:
class AttentionMultiHead:
  def __init__(self, dims, nHeads, rangeQ=(-1,1),rangeK=(-1,1),rangeV=(-1,1), rangeW=(-1,1)):
    head_size = dims//nHeads
    # Corrected: AttentionHead expects dims1 (input_dim) and dims2 (output_dim=head_size)
    self.heads = [AttentionHead(dims, head_size, rangeQ=rangeQ,rangeK=rangeK,rangeV=rangeV) for _ in range(nHeads)]
    # w_0 should project from (nHeads * head_size) to dims
    self.w_0 = Values((rangeW[0]-rangeW[1])*np.random.rand(nHeads * head_size, dims)+rangeW[1])
    self.dims = dims # Store dims for potential future use
    self.nHeads = nHeads # Store nHeads
    self.head_size = head_size # Store head_size

  def __call__(self,x):
    # x is expected to be (batch_size, sequence_length, embedding_dim)
    outs = [h(x) for h in self.heads] # Each h(x) will be (batch_size, sequence_length, head_size)

    # Get the dynamic batch and sequence dimensions from the input x
    batch_and_seq_dims = x.shape[:-1] # This will be (batch_size, sequence_length)

    # Calculate the total concatenated dimension
    concatenated_feature_dim = self.nHeads * self.head_size

    # Initialize concatenated_output with the correct shape: (batch_size, sequence_length, concatenated_feature_dim)
    concatenated_output_vals = np.zeros(batch_and_seq_dims + (concatenated_feature_dim,))
    concatenated_output = Values(concatenated_output_vals)

    current_idx = 0
    for head_output in outs:
      # head_output shape: (batch_size, sequence_length, head_size)
      output_width = head_output.shape[-1] # This is self.head_size
      # Assign the head's output to the corresponding slice of the concatenated_output
      # Using ellipsis `...` to handle any number of leading batch/sequence dimensions
      concatenated_output[..., current_idx : current_idx + output_width] = head_output
      current_idx += output_width

    # Apply the final linear layer (projection) using w_0
    # concatenated_output has shape (batch_size, sequence_length, nHeads * head_size)
    # self.w_0 has shape (nHeads * head_size, dims)
    # Resulting shape will be (batch_size, sequence_length, dims)
    final_output = concatenated_output @ self.w_0
    return final_output

  def params(self):
    all_params = []
    for h in self.heads:
      all_params.extend(h.params())
    all_params.append(self.w_0)
    return all_params

In [8]:
# Test for AttentionMultiHead and Backpropagation

# Define parameters
dims = 64
n_heads = 4
seq_len = 10

# Instantiate MultiHead Attention
multi_head_attention = AttentionMultiHead(dims=dims, nHeads=n_heads)

# Create a dummy input
x_input_multi = Values(np.random.rand(seq_len, dims))

# Get output from MultiHead Attention
output_multi = multi_head_attention(x_input_multi)

print(f"MultiHead Attention Input shape: {x_input_multi.shape}")
print(f"MultiHead Attention Output shape: {output_multi.shape}")

# --- Backpropagation Test for MultiHead Attention ---
print('\n--- MultiHead Attention Backpropagation Test ---')

# Create a dummy target output for loss calculation
y_true_multi = Values(np.random.rand(seq_len, dims))

# Calculate MSE loss
loss_multi = mse_loss(output_multi, y_true_multi)

# Perform backward pass
loss_multi.backward()

# Print gradients for all parameters using the params() method
print("\n--- Gradients from params() method ---")
all_params = multi_head_attention.params()

# The structure of `all_params` is now a flattened list of Values objects
for i, param in enumerate(all_params):
    param_name = ""
    if i < len(all_params) - 1: # These are head parameters
        head_idx = i // 3
        param_type = ["w_q", "w_k", "w_v"][i % 3]
        param_name = f"Head {head_idx} Parameter ({param_type})"
    else: # This is the last parameter, w_0
        param_name = "Output Projection (w_0)"

    print(f"  {param_name} shape: {param.grad.shape}, first few values:\n{param.grad[:2, :2]}")

MultiHead Attention Input shape: (10, 64)
MultiHead Attention Output shape: (10, 64)

--- MultiHead Attention Backpropagation Test ---

--- Gradients from params() method ---
  Head 0 Parameter (w_q) shape: (64, 16), first few values:
[[ 0.48781152 -0.56968823]
 [ 0.35827683 -0.4448084 ]]
  Head 0 Parameter (w_k) shape: (64, 16), first few values:
[[0.35626046 0.37746888]
 [0.5575907  0.35912759]]
  Head 0 Parameter (w_v) shape: (64, 16), first few values:
[[-0.03204474 -0.37149421]
 [-0.04454796 -1.1234413 ]]
  Head 1 Parameter (w_q) shape: (64, 16), first few values:
[[-0.07754746  0.06138669]
 [-0.03962482  0.01996259]]
  Head 1 Parameter (w_k) shape: (64, 16), first few values:
[[ 0.69774189  0.03814159]
 [ 0.56246466 -0.00679746]]
  Head 1 Parameter (w_v) shape: (64, 16), first few values:
[[-0.99005518  0.37549211]
 [-0.40338725  0.17798415]]
  Head 2 Parameter (w_q) shape: (64, 16), first few values:
[[0.35883046 0.46382   ]
 [0.33418606 0.27278666]]
  Head 2 Parameter (w_k) sha

In [9]:
# Test for AttentionMultiHead and Model Training

# Define parameters
dims = 64
n_heads = 4
seq_len = 10
epochs = 5

# Instantiate MultiHead Attention and wrap it in a Model
m = Model([AttentionMultiHead(dims=dims, nHeads=n_heads)], loss_fn=mse_loss)

# Create dummy input data for training and validation
x_train = Values(np.random.rand(seq_len * epochs, dims))
y_train = Values(np.random.rand(seq_len * epochs, dims))
x_val = Values(np.random.rand(seq_len, dims))
y_val = Values(np.random.rand(seq_len, dims))

print("Starting Model training...")
# Call the train method
m.train(epochs=epochs, x_t=x_train, y_t=y_train, x_v=x_val, y_v=y_val)
print("Model training complete.")

# Print the training and validation losses
print(f"Final Training Loss: {m.train_loss}")
print(f"Final Validation Loss: {m.val_loss}")


Starting Model training...
epoch: 0 	 loss: 92.63838902500217
epoch: 1 	 loss: 53.904129806384525
epoch: 2 	 loss: 42.07886893004049
epoch: 3 	 loss: 37.029823802269185
epoch: 4 	 loss: 34.47258489280041
epoch: 5 	 loss: 32.98386035412554
Model training complete.
Final Training Loss: [(0.0, array(104.39062893)), (1.0, array(45.58092944)), (2.0, array(28.02357031)), (3.0, array(20.25888932)), (4.0, array(16.0729699))]
Final Validation Loss: [(0, array(92.63838903)), (1, array(53.90412981)), (2, array(42.07886893)), (3, array(37.0298238)), (4, array(34.47258489)), (5, array(32.98386035))]


In [10]:
#Embedding, include positional embedding option

In [11]:
import numpy as np
from rb_deeplearning_lib import Values # Ensure Values is imported in this scope if the cell were standalone.
                                     # It is imported in u3RHlJRyDl4U.

class Embedding():
  def __init__(self, elements, dims, rangeE=(-1,1)):
    self.elements = elements # Corrected: store elements on self
    self.len = len(elements)
    self.encoder = {el:i for i, el in enumerate(elements)}
    self.decoder = dict(enumerate(elements))
    self.dims = dims

    embeddings = []
    for i in range(self.len):
      embeddings.append(Values((rangeE[0]-rangeE[1])*np.random.rand(dims)+rangeE[1]))
    self.embed = embeddings

  def encode(self, x_elements_flat):
    # x_elements_flat is assumed to be a 1D iterable of elements (e.g., ['a', 'b', 'c'])
    enc = self.encoder
    out = [enc.get(el) for el in x_elements_flat]
    return np.array(out) # Return a numpy array of encoded indices

  def decode(self, y_indices_flat):
    # y_indices_flat is assumed to be a 1D iterable of integer keys
    dec = self.decoder
    out = [dec.get(key) for key in y_indices_flat]
    return out

  def __call__(self, x):
    # Determine the actual data to process and its conceptual shape
    elements_to_process = None
    original_sequence_shape = None # Shape of the input *before* embedding dimension is added

    if isinstance(x, str):
      elements_to_process = list(x) # Treat string as sequence of characters
      original_sequence_shape = (len(x),)
      is_encoded_input = False
    elif isinstance(x, Values):
        x_data = x.vals
        original_sequence_shape = x_data.shape
        elements_to_process = x_data.flatten()
        # Check if elements are already integer indices
        is_encoded_input = np.issubdtype(x_data.dtype, np.integer) or \
                           (x_data.size > 0 and isinstance(elements_to_process[0], int))
    else: # Handles lists, tuples, numpy arrays of elements or integers
        x_data = np.asarray(x)
        original_sequence_shape = x_data.shape
        elements_to_process = x_data.flatten()
        # Check if elements are already integer indices
        is_encoded_input = np.issubdtype(x_data.dtype, np.integer) or \
                           (x_data.size > 0 and isinstance(elements_to_process[0], int))

    if is_encoded_input:
      encoded_indices_flat = elements_to_process
    else:
      encoded_indices_flat = self.encode(elements_to_process)

    num_elements_flat = len(encoded_indices_flat)
    embeded_flat_vals = np.zeros((num_elements_flat, self.dims))

    for i in range(num_elements_flat):
      current_index = encoded_indices_flat[i]
      if current_index is None:
          # Find the original element from elements_to_process at index i
          element_that_failed = elements_to_process[i]
          raise ValueError(f"Element '{element_that_failed}' not found in embedding vocabulary. "
                           f"Vocabulary: {list(self.encoder.keys())}")
      embeded_flat_vals[i,:] = self.embed[current_index].vals

    output_embedding_shape = original_sequence_shape + (self.dims,)
    embeded = Values(embeded_flat_vals.reshape(output_embedding_shape))

    return embeded

  def params(self):
    return self.embed

In [12]:
em = Embedding(['a','b','c','d','e'], 2)
em.embed

[vals: array([-0.16800458,  0.37095843])
 grads: array([0., 0.]),
 vals: array([ 0.79890707, -0.70596751])
 grads: array([0., 0.]),
 vals: array([0.1216092 , 0.60139493])
 grads: array([0., 0.]),
 vals: array([ 0.61848764, -0.870348  ])
 grads: array([0., 0.]),
 vals: array([-0.83013304, -0.77055825])
 grads: array([0., 0.])]

In [13]:
em('abcd')

vals: array([[-0.16800458,  0.37095843],
       [ 0.79890707, -0.70596751],
       [ 0.1216092 ,  0.60139493],
       [ 0.61848764, -0.870348  ]])
grads: array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

In [14]:
em(['a','b','c','d'])

vals: array([[-0.16800458,  0.37095843],
       [ 0.79890707, -0.70596751],
       [ 0.1216092 ,  0.60139493],
       [ 0.61848764, -0.870348  ]])
grads: array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

In [15]:
em([['a','b','c','d'],['a','b','c','d']])

vals: array([[[-0.16800458,  0.37095843],
        [ 0.79890707, -0.70596751],
        [ 0.1216092 ,  0.60139493],
        [ 0.61848764, -0.870348  ]],

       [[-0.16800458,  0.37095843],
        [ 0.79890707, -0.70596751],
        [ 0.1216092 ,  0.60139493],
        [ 0.61848764, -0.870348  ]]])
grads: array([[[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]],

       [[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]]])

In [16]:
import numpy as np

# Test for Embedding Backpropagation

# Define parameters
elements = ['a', 'b', 'c', 'd', 'e']
dims = 2 # Embedding dimensions
seq_len = 4 # Length of input sequence 'abcd'

# Instantiate Embedding
embedding_layer = Embedding(elements=elements, dims=dims)

# Create a dummy input for embedding
# This input uses 'a', 'b', 'c', 'd'
x_input_embed = 'abcd'

# Get output from Embedding layer
embed_output = embedding_layer(x_input_embed)

print(f"Embedding Input (encoded): {embedding_layer.encode(x_input_embed)}")
print(f"Embedding Output shape: {embed_output.shape}")

# Create a dummy target output for loss calculation
y_true_embed = Values(np.random.rand(seq_len, dims))

# Calculate MSE loss
loss_embed = mse_loss(embed_output, y_true_embed)

# Perform backward pass
loss_embed.backward()

print("\n--- Embedding Backpropagation Test ---")

# Check gradients for each individual embedding vector
print("Checking gradients for individual embedding vectors:")
for i, embed_vec_val in enumerate(embedding_layer.embed):
    element = embedding_layer.decoder[i]
    if embed_vec_val.grad is not None:
        # Check if the element was part of the input sequence 'abcd'
        if element in x_input_embed:
            # Expected to have non-zero gradients
            is_grad_zero = np.all(embed_vec_val.grad == 0)
            print(f"  Embedding for '{element}' (Index {i}): Grad present. All zeros: {is_grad_zero}")
            if is_grad_zero:
                print(f"    (ERROR: Expected non-zero gradient for '{element}', but got all zeros.)")
            else:
                print(f"    First few grad values:\n{embed_vec_val.grad.flatten()[:min(2, dims)]}...") # Print a few values
        else:
            # Expected to have zero gradients
            is_grad_zero = np.all(embed_vec_val.grad == 0)
            print(f"  Embedding for '{element}' (Index {i}): Grad present. All zeros: {is_grad_zero}")
            if not is_grad_zero:
                print(f"    (ERROR: Expected zero gradient for '{element}', but got non-zero values.)")
                print(f"    First few grad values:\n{embed_vec_val.grad.flatten()[:min(2, dims)]}...") # Print a few values

    else:
        print(f"  Embedding for '{element}' (Index {i}): Grad is None. (ERROR: Gradient should be initialized to zeros)")


Embedding Input (encoded): [0 1 2 3]
Embedding Output shape: (4, 2)

--- Embedding Backpropagation Test ---
Checking gradients for individual embedding vectors:
  Embedding for 'a' (Index 0): Grad present. All zeros: True
    (ERROR: Expected non-zero gradient for 'a', but got all zeros.)
  Embedding for 'b' (Index 1): Grad present. All zeros: True
    (ERROR: Expected non-zero gradient for 'b', but got all zeros.)
  Embedding for 'c' (Index 2): Grad present. All zeros: True
    (ERROR: Expected non-zero gradient for 'c', but got all zeros.)
  Embedding for 'd' (Index 3): Grad present. All zeros: True
    (ERROR: Expected non-zero gradient for 'd', but got all zeros.)
  Embedding for 'e' (Index 4): Grad present. All zeros: True


In [17]:
x = np.zeros((4,3))
x[1]

array([0., 0., 0.])

In [21]:
from rb_deeplearning_lib import Model, Dense, mse_loss , AttentionMultiHead, Embedding

In [22]:
# Define model parameters
vocabulary = ['<PAD>', '<START>', '<END>', 'a', 'b', 'c', 'd', 'e'] # Example vocabulary
embedding_dim = 64 # Dimension for embeddings and attention
num_heads = 4      # Number of attention heads

# Instantiate the layers
embedding_layer = Embedding(elements=vocabulary, dims=embedding_dim)
attention_block = AttentionMultiHead(dims=embedding_dim, nHeads=num_heads)
# Dense layer: layNum (number of layers), inL (input features), midL (hidden features), outL (output features)
# For simplicity, let's use a single dense layer that projects the attention output back to embedding_dim.
dense_layer = Dense(layNum=1, inL=embedding_dim, midL=embedding_dim * 2, outL=embedding_dim, activ='relu')

# Combine into a Model
m = Model(blocks=[embedding_layer, attention_block, dense_layer], loss_fn=mse_loss)

print("Model 'm' created successfully with Embedding, AttentionMultiHead, and Dense layers.")

Model 'm' created successfully with Embedding, AttentionMultiHead, and Dense layers.


In [23]:
# Generate dummy training data for the model

# Define training parameters
epochs = 10
batch_size = 8
seq_len = 5 # Sequence length of input tokens

# Get vocabulary from the embedding layer
vocabulary_size = embedding_layer.len

# Create dummy input token IDs (batch_size, seq_len)
# Tokens are integers corresponding to indices in the vocabulary
x_train_tokens = Values(np.random.randint(0, vocabulary_size, size=(epochs * batch_size, seq_len)))
x_val_tokens = Values(np.random.randint(0, vocabulary_size, size=(batch_size, seq_len)))

# Create dummy target data (batch_size, seq_len, embedding_dim)
# The output of the model will be embeddings, so y should match that shape
y_train = Values(np.random.rand(epochs * batch_size, seq_len, embedding_dim))
y_val = Values(np.random.rand(batch_size, seq_len, embedding_dim))

print(f"x_train_tokens shape: {x_train_tokens.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_val_tokens shape: {x_val_tokens.shape}")
print(f"y_val shape: {y_val.shape}")

print("\nStarting model training...")
m.train(epochs=epochs, x_t=x_train_tokens, y_t=y_train, x_v=x_val_tokens, y_v=y_val, batch_size=batch_size)
print("Model training complete.")

print(f"\nFinal Training Loss: {m.train_loss[-1][1]:.4f}")
print(f"Final Validation Loss: {m.val_loss[-1][1]:.4f}")

x_train_tokens shape: (80, 5)
y_train shape: (80, 5, 64)
x_val_tokens shape: (8, 5)
y_val shape: (8, 5, 64)

Starting model training...
epoch: 0 	 loss: 28.52207534281134
epoch: 1 	 loss: 2.1221155363214406
epoch: 2 	 loss: 1.112178170753807
epoch: 3 	 loss: 0.695448002507225
epoch: 4 	 loss: 0.5155434594791938
epoch: 5 	 loss: 0.37959233086365574
epoch: 6 	 loss: 0.3194659063345305
epoch: 7 	 loss: 0.28781038333453857
epoch: 8 	 loss: 0.24091886239258825
epoch: 9 	 loss: 0.21100190316447573
epoch: 10 	 loss: 0.19863313642595162
Model training complete.

Final Training Loss: 0.2100
Final Validation Loss: 0.1986
